In [1]:
import numpy as np
from plasmapy.formulary import Debye_length
from astropy import units as u

# ── Physical Constants ─────────────────────────────────────
e    = 1.602e-19
eps0 = 8.854e-12
m_p  = 1.6726e-27
k_B  = 1.381e-23

# ── Updated Plasma Conditions ──────────────────────────────
T_eV     = 0.1                 # lowered from 1.0 eV
T_K      = T_eV * 11604.5
n_e      = 1e21                # raised from 1e19 m⁻³

lambda_D = Debye_length(T_eV * u.eV, n_e * u.m**-3).to(u.m).value

# ── Normalized Coulomb Prefactor ───────────────────────────
A_SI   = e**2 / (4 * np.pi * eps0)
A_norm = A_SI / (k_B * T_K * lambda_D)
q_eff  = np.sqrt(A_norm)

# ── Full Mass Ratio ────────────────────────────────────────
mass_proton   = 1.0
mass_electron = 1.0 / 1836.0

# ── Simulation Parameters ─────────────────────────────────
N_protons   = 150
N_electrons = 150
box_size    = 10.0
cutoff      = 4.0
timestep    = 0.00005
damping_p   = 0.05
damping_e   = 0.005
steps       = 10000
dump_freq   = 50

# ── Coupling Parameter ─────────────────────────────────────
r_avg = (3 / (4 * np.pi * (N_protons / box_size**3))) ** (1/3)
gamma = A_norm / r_avg

print(f"Temperature         : {T_K:.1f} K  ({T_eV} eV)")
print(f"Number Density      : {n_e:.2e} m⁻³")
print(f"Debye Length        : {lambda_D:.4e} m")
print(f"Yukawa A (norm)     : {A_norm:.4f}")
print(f"Effective charge    : ±{q_eff:.6f}")
print(f"Coupling Γ          : {gamma:.4f}  (was 0.0004 before)")
print(f"Frames to render    : {steps // dump_freq}")

Temperature         : 1160.5 K  (0.1 eV)
Number Density      : 1.00e+21 m⁻³
Debye Length        : 7.4339e-08 m
Yukawa A (norm)     : 0.1936
Effective charge    : ±0.440016
Coupling Γ          : 0.1658  (was 0.0004 before)
Frames to render    : 200


In [2]:
from lammps import lammps

lmp = lammps()

lmp.commands_string(f"""
# ── Setup ──────────────────────────────────────────────────
units         lj
atom_style    charge
boundary      p p p

# ── Box & Atoms ────────────────────────────────────────────
region        box block 0 {box_size} 0 {box_size} 0 {box_size}
create_box    2 box

# Larger overlap distance to prevent particles spawning too close
create_atoms  1 random {N_protons}   12345 box overlap 1.0 maxtry 5000
create_atoms  2 random {N_electrons} 67890 box overlap 1.0 maxtry 5000

# ── Masses & Charges ───────────────────────────────────────
mass          1 {mass_proton}
mass          2 {mass_electron:.8f}

set           type 1 charge +{q_eff:.6f}
set           type 2 charge -{q_eff:.6f}

# ── Neighbor List ──────────────────────────────────────────
neigh_modify  one 5000 delay 0 every 1 check yes

# ── Stage 1: Energy Minimization ──────────────────────────
# Push overlapping particles apart before any dynamics
pair_style    soft 1.0
pair_coeff    * * 10.0
minimize      1.0e-4 1.0e-6 1000 10000

# ── Stage 2: Gentle NVE warm-up with soft potential ────────
reset_timestep 0
velocity      all create 0.1 11111 dist gaussian
fix           warmup all nve/limit 0.01
timestep      0.00001
run           2000
unfix         warmup

# ── Stage 3: Gradual velocity ramp with soft potential ─────
fix           ramp all nvt temp 0.1 1.0 0.05
run           3000
unfix         ramp

# ── Stage 4: Switch to Coulomb ─────────────────────────────
pair_style    coul/cut {cutoff}
pair_coeff    * *

neigh_modify  one 5000 delay 0 every 1 check yes
reset_timestep 0

# ── Separate Thermostats ───────────────────────────────────
group         protons   type 1
group         electrons type 2

velocity      protons   create 1.0 11111 dist gaussian
velocity      electrons create 1.0 22222 dist gaussian

fix           1 protons   nvt temp 1.0 1.0 {damping_p}
fix           2 electrons nvt temp 1.0 1.0 {damping_e}

# Prevent center-of-mass drift
fix           3 all momentum 100 linear 1 1 1

# ── Dump ───────────────────────────────────────────────────
dump          1 all custom {dump_freq} plasma.dump id type x y z
dump_modify   1 sort id

# ── Production Run ─────────────────────────────────────────
timestep      {timestep}
thermo        {dump_freq}
run           {steps}
""")

lmp.close()
print("✓ LAMMPS simulation complete — plasma.dump written")

LAMMPS (29 Aug 2024)
OMP_NUM_THREADS environment is not set. Defaulting to 1 thread. (src/comm.cpp:98)
  using 1 OpenMP thread(s) per MPI task
Created orthogonal box = (0 0 0) to (10 10 10)
  1 by 1 by 1 MPI processor grid
Created 150 atoms
  using lattice units in orthogonal box = (0 0 0) to (10 10 10)
  create_atoms CPU = 0.001 seconds
Created 150 atoms
  using lattice units in orthogonal box = (0 0 0) to (10 10 10)
  create_atoms CPU = 0.001 seconds
Setting atom values ...
  150 settings made for charge
Setting atom values ...
  150 settings made for charge
Generated 0 of 1 mixed pair_coeff terms from geometric mixing rule
Neighbor list info ...
  update: every = 1 steps, delay = 0 steps, check = yes
  max neighbors/atom: 5000, page size: 100000
  master list distance cutoff = 1.3
  ghost atom cutoff = 1.3
  binsize = 0.65, bins = 16 16 16
  1 neighbor lists, perpetual/occasional/extra = 1 0 0
  (1) pair soft, perpetual
      attributes: half, newton on
      pair build: half/bin/at

In [3]:
def parse_lammps_dump(filepath):
    frames_pos   = []
    frames_types = []

    with open(filepath, "r") as f:
        lines = f.readlines()

    i = 0
    while i < len(lines):
        if "ITEM: TIMESTEP" in lines[i]:
            i += 2
            i += 2
            i += 4  # skip box bounds
            i += 1  # skip ITEM: ATOMS header

            positions = []
            types     = []
            while i < len(lines) and "ITEM:" not in lines[i]:
                parts = lines[i].split()
                types.append(int(parts[1]))
                x, y, z = float(parts[2]), float(parts[3]), float(parts[4])
                positions.append([x, y, z])
                i += 1

            frames_pos.append(np.array(positions))
            frames_types.append(np.array(types))
        else:
            i += 1

    return frames_pos, frames_types

frames_pos, frames_types = parse_lammps_dump("plasma.dump")

print(f"✓ Parsed {len(frames_pos)} frames")
print(f"✓ Protons   per frame : {np.sum(frames_types[0] == 1)}")
print(f"✓ Electrons per frame : {np.sum(frames_types[0] == 2)}")

✓ Parsed 201 frames
✓ Protons   per frame : 150
✓ Electrons per frame : 150


In [4]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.gridspec as gridspec
from scipy.ndimage import gaussian_filter
import numpy as np

# ── Skip frame 0 (pre-run initial state) ──────────────────
frames_pos_use   = frames_pos[1:]
frames_types_use = frames_types[1:]
n_frames         = len(frames_pos_use)

# ── RDF Setup ─────────────────────────────────────────────
n_bins    = 80
r_max     = cutoff * 0.9
r_bins    = np.linspace(0.1, r_max, n_bins + 1)
r_centers = 0.5 * (r_bins[:-1] + r_bins[1:])
rdf_accum = np.zeros(n_bins)
rdf_count = 0

# ── Radial Ring Image Setup ────────────────────────────────
ring_res  = 300
ring_half = ring_res // 2
yy, xx    = np.mgrid[-ring_half:ring_half, -ring_half:ring_half]
pixel_r   = np.sqrt(xx**2 + yy**2) / ring_half * r_max

# ── Minimum Image Convention ───────────────────────────────
def min_image(delta, box):
    return delta - box * np.round(delta / box)

# ── Precompute RDF across all frames ──────────────────────
rdf_history = []

print("Precomputing RDF across all frames...")
for frame_idx in range(n_frames):
    pos   = frames_pos_use[frame_idx]
    types = frames_types_use[frame_idx]

    p_pos = pos[types == 1]
    e_pos = pos[types == 2]

    frame_rdf = np.zeros(n_bins)
    for p in p_pos:
        for e in e_pos:
            dr   = min_image(e - p, box_size)
            dist = np.linalg.norm(dr)
            if 0.1 < dist < r_max:
                bin_idx = np.searchsorted(r_bins, dist) - 1
                if 0 <= bin_idx < n_bins:
                    frame_rdf[bin_idx] += 1

    n_e_density = N_electrons / box_size**3
    for b in range(n_bins):
        shell_vol = (4/3) * np.pi * (r_bins[b+1]**3 - r_bins[b]**3)
        expected  = n_e_density * shell_vol * N_protons
        frame_rdf[b] = frame_rdf[b] / expected if expected > 0 else 0

    rdf_accum += frame_rdf
    rdf_count += 1
    rdf_history.append(rdf_accum.copy() / rdf_count)

print(f"✓ Precomputed {n_frames} frames")

# ── Build radial ring images ───────────────────────────────
def make_ring_image(rdf):
    img = np.zeros((ring_res, ring_res))
    for i in range(ring_res):
        for j in range(ring_res):
            r = pixel_r[i, j]
            if r < r_max:
                bin_idx = np.searchsorted(r_bins, r) - 1
                if 0 <= bin_idx < n_bins:
                    img[i, j] = rdf[bin_idx]
    return img

print("Building radial ring images...")
ring_history = [make_ring_image(rdf_history[i]) for i in range(n_frames)]
print("✓ Ring images ready")

# ── Derived stats for the panel ───────────────────────────
r_avg      = (3 / (4 * np.pi * (N_protons / box_size**3))) ** (1/3)
gamma      = A_norm / r_avg
plasma_freq = np.sqrt(N_protons / box_size**3 / mass_proton)  # normalized
rdf_final  = rdf_history[-1]
peak_val   = rdf_final.max()
peak_r     = r_centers[np.argmax(rdf_final)]

# ── Build Figure Layout ───────────────────────────────────
fig = plt.figure(figsize=(15, 10), facecolor="#0a0a1a")

# Outer grid: top (simulation panels) + bottom (stats bar)
outer_gs = gridspec.GridSpec(2, 1,
                             height_ratios=[5, 1],
                             hspace=0.08,
                             figure=fig)

# Top section: 3D view + RDF + ring
top_gs = gridspec.GridSpecFromSubplotSpec(2, 2,
                                          subplot_spec=outer_gs[0],
                                          width_ratios=[1.3, 1],
                                          hspace=0.45, wspace=0.35)

ax3d    = fig.add_subplot(top_gs[:, 0], projection='3d')
ax_rdf  = fig.add_subplot(top_gs[0, 1])
ax_ring = fig.add_subplot(top_gs[1, 1])

# Bottom section: stats panel
ax_stats = fig.add_subplot(outer_gs[1])
ax_stats.set_facecolor("#0d0d1f")
ax_stats.axis('off')
for spine in ax_stats.spines.values():
    spine.set_edgecolor('#334')

for ax in [ax_rdf, ax_ring]:
    ax.set_facecolor("#0d0d2b")
    for spine in ax.spines.values():
        spine.set_edgecolor('#444466')
    ax.tick_params(colors='#aaaacc', labelsize=7)
    ax.xaxis.label.set_color('#aaaacc')
    ax.yaxis.label.set_color('#aaaacc')
    ax.title.set_color('white')

# ── Initialize ring image ─────────────────────────────────
ring_img = ax_ring.imshow(
    gaussian_filter(ring_history[0], sigma=3),
    extent=[-r_max, r_max, -r_max, r_max],
    origin='lower', cmap='inferno',
    aspect='equal', vmin=0, vmax=2.5
)
theta = np.linspace(0, 2 * np.pi, 300)
ax_ring.plot(np.cos(theta), np.sin(theta),
             color='cyan', linewidth=0.8,
             linestyle='--', alpha=0.6, label='r = 1 λ_D')
ax_ring.legend(fontsize=6, facecolor='#1a1a2e',
               labelcolor='white', framealpha=0.7, loc='upper right')

cbar = fig.colorbar(ring_img, ax=ax_ring, fraction=0.046, pad=0.04)
cbar.ax.tick_params(colors='#aaaacc', labelsize=6)
cbar.set_label('g(r)', color='#aaaacc', fontsize=7)

# ── Stats panel content ───────────────────────────────────
col_style = dict(color='#aaaacc', fontsize=8,
                 fontfamily='monospace', transform=ax_stats.transAxes,
                 va='center')
header_style = dict(color='white', fontsize=8, fontweight='bold',
                    fontfamily='monospace', transform=ax_stats.transAxes,
                    va='center')

# Divider line at top of stats panel
ax_stats.axhline(y=0.95, color='#334477', linewidth=0.8)

# Column headers
ax_stats.text(0.01,  0.72, "── PARTICLES ──",         **header_style)
ax_stats.text(0.21,  0.72, "── PLASMA CONDITIONS ──",  **header_style)
ax_stats.text(0.50,  0.72, "── DEBYE PARAMETERS ──",   **header_style)
ax_stats.text(0.74,  0.72, "── SIMULATION ──",         **header_style)

# Column 1 — Particles
ax_stats.text(0.01, 0.45, f"Protons        : {N_protons}",     **col_style)
ax_stats.text(0.01, 0.20, f"Electrons      : {N_electrons}",   **col_style)

# Column 2 — Plasma conditions
ax_stats.text(0.21, 0.45, f"Temperature    : {T_eV} eV  ({T_K:.0f} K)",  **col_style)
ax_stats.text(0.21, 0.20, f"Density        : {n_e:.2e} m⁻³",             **col_style)

# Column 3 — Debye parameters
ax_stats.text(0.50, 0.45, f"Debye Length   : {lambda_D:.4e} m",           **col_style)
ax_stats.text(0.50, 0.20, f"Coupling Γ     : {gamma:.5f}  (weakly coupled)" if gamma < 1
                           else f"Coupling Γ     : {gamma:.5f}  (strongly coupled)", **col_style)

# Column 4 — Simulation inputs
ax_stats.text(0.74, 0.45, f"MD Steps       : {steps}  |  Timestep : {timestep}",  **col_style)
ax_stats.text(0.74, 0.20, f"Box Size       : {box_size} λ_D  |  Cutoff : {cutoff} λ_D", **col_style)

# ── Dynamic stats (update each frame) ────────────────────
rdf_peak_text = ax_stats.text(0.50, -0.15, "", fontsize=8, color='mediumpurple',
                               fontfamily='monospace', transform=ax_stats.transAxes,
                               va='center', ha='center', clip_on=False)

def update(frame_idx):
    # ── Left: 3D particle positions ───────────────────────
    ax3d.cla()
    ax3d.set_facecolor("#0a0a1a")

    pos   = frames_pos_use[frame_idx]
    types = frames_types_use[frame_idx]
    p_pos = pos[types == 1]
    e_pos = pos[types == 2]

    ax3d.scatter(p_pos[:, 0], p_pos[:, 1], p_pos[:, 2],
                 c='crimson', s=60, alpha=0.95,
                 edgecolors='darkred', linewidths=0.4, label='Protons')
    ax3d.scatter(e_pos[:, 0], e_pos[:, 1], e_pos[:, 2],
                 c='dodgerblue', s=10, alpha=0.7,
                 edgecolors='navy', linewidths=0.2, label='Electrons')

    ax3d.set_xlim(0, box_size); ax3d.set_ylim(0, box_size); ax3d.set_zlim(0, box_size)
    ax3d.set_xlabel("x (λ_D)", color='white', fontsize=7)
    ax3d.set_ylabel("y (λ_D)", color='white', fontsize=7)
    ax3d.set_zlabel("z (λ_D)", color='white', fontsize=7)
    ax3d.tick_params(colors='white', labelsize=6)
    ax3d.set_facecolor("#0a0a1a")
    ax3d.legend(loc='upper left', fontsize=7,
                facecolor='#1a1a2e', labelcolor='white', framealpha=0.7)
    ax3d.set_title(
        f"Particle Positions  |  {N_protons}p + {N_electrons}e\n"
        f"Frame {frame_idx + 1}/{n_frames}",
        color='white', fontsize=9)

    # ── Right top: Running RDF ─────────────────────────────
    ax_rdf.cla()
    ax_rdf.set_facecolor("#0d0d2b")
    rdf = rdf_history[frame_idx]
    ax_rdf.plot(r_centers, rdf, color='mediumpurple', linewidth=1.5)
    ax_rdf.axhline(y=1.0, color='gray', linestyle='--',
                   linewidth=0.8, alpha=0.6, label='Random (g=1)')
    ax_rdf.fill_between(r_centers, 1.0, rdf,
                        where=(rdf > 1), alpha=0.25,
                        color='mediumpurple', label='Electron excess')
    ax_rdf.fill_between(r_centers, 1.0, rdf,
                        where=(rdf < 1), alpha=0.25,
                        color='tomato', label='Electron deficit')
    ax_rdf.set_xlim(0, r_max)
    ax_rdf.set_ylim(0, max(2.5, rdf.max() * 1.1))
    ax_rdf.set_xlabel("r (λ_D)", color='#aaaacc', fontsize=8)
    ax_rdf.set_ylabel("g(r)  electron-proton", color='#aaaacc', fontsize=8)
    ax_rdf.set_title("Radial Distribution Function\n(accumulating over time)",
                     color='white', fontsize=8)
    ax_rdf.legend(fontsize=6, facecolor='#1a1a2e',
                  labelcolor='white', framealpha=0.7)
    ax_rdf.tick_params(colors='#aaaacc', labelsize=7)
    for spine in ax_rdf.spines.values():
        spine.set_edgecolor('#444466')

    # ── Right bottom: Radial ring image ───────────────────
    smoothed = gaussian_filter(ring_history[frame_idx], sigma=3)
    ring_img.set_data(smoothed)
    ring_img.set_clim(vmin=0, vmax=max(smoothed.max(), 2.5))
    ax_ring.set_xlabel("Δx (λ_D)", color='#aaaacc', fontsize=8)
    ax_ring.set_ylabel("Δy (λ_D)", color='#aaaacc', fontsize=8)
    ax_ring.set_title(
        "Debye Screening Cloud\n(radial electron density, time-averaged)",
        color='white', fontsize=8)

    # ── Stats panel: update live RDF peak ─────────────────
    current_peak = rdf_history[frame_idx].max()
    current_peak_r = r_centers[np.argmax(rdf_history[frame_idx])]
    rdf_peak_text.set_text(
        f"[ Live ]  RDF Peak:  g(r) = {current_peak:.3f}  |  "
        f"Peak at r = {current_peak_r:.3f} λ_D  |  "
        f"Frame {frame_idx + 1}/{n_frames}"
    )

    fig.patch.set_facecolor("#0a0a1a")

ani = animation.FuncAnimation(fig, update, frames=n_frames, interval=60)

writer = animation.FFMpegWriter(fps=20, bitrate=2400)
ani.save("plasma_screening.mp4", writer=writer)
plt.close()
print("✓ Saved plasma_screening.mp4")

Precomputing RDF across all frames...
✓ Precomputed 200 frames
Building radial ring images...
✓ Ring images ready
✓ Saved plasma_screening.mp4


In [5]:
import os

size = os.path.getsize("plasma.dump")
print(f"plasma.dump size: {size / 1024:.1f} KB")

# Check how many frames were actually written
with open("plasma.dump") as f:
    frame_count = sum(1 for line in f if "ITEM: TIMESTEP" in line)
print(f"Frames written: {frame_count}  (expected: {steps // dump_freq})")

plasma.dump size: 1792.0 KB
Frames written: 201  (expected: 200)


In [6]:
# ── Sanity Check Cell ─────────────────────────────────────
from scipy.spatial.distance import cdist

print("=== Simulation Sanity Checks ===\n")

# 1. Average electron-proton distance vs Debye length
avg_ep_dists = []
for frame_idx in range(0, n_frames, 10):   # sample every 10 frames
    pos   = frames_pos_use[frame_idx]
    types = frames_types_use[frame_idx]
    p_pos = pos[types == 1]
    e_pos = pos[types == 2]

    dists = cdist(e_pos, p_pos).min(axis=1)   # nearest proton for each electron
    avg_ep_dists.append(np.mean(dists))

mean_dist = np.mean(avg_ep_dists)
print(f"1. Avg electron-proton nearest distance : {mean_dist:.3f} λ_D")
print(f"   Expected (screened)                  : < 1.0 λ_D")
print(f"   {'✅ Looks good' if mean_dist < 1.5 else '⚠️  Electrons may not be clustering'}\n")

# 2. Coupling parameter Γ — tells us how strongly coupled the plasma is
# Γ = potential energy / kinetic energy per particle
# Γ << 1 → weakly coupled (Debye-Hückel valid)
# Γ >> 1 → strongly coupled (more complex physics)
r_avg  = (3 / (4 * np.pi * (N_protons / box_size**3))) ** (1/3)  # Wigner-Seitz radius
gamma  = A_norm / r_avg
print(f"2. Coupling parameter Γ                 : {gamma:.4f}")
print(f"   Γ << 1 → weakly coupled (Debye-Hückel valid)")
print(f"   Γ >> 1 → strongly coupled (different regime)")
print(f"   {'✅ Weakly coupled — Debye screening applies' if gamma < 1 else '⚠️  Strongly coupled — results may differ from simple Debye theory'}\n")

# 3. Check particle count is stable across frames
counts = [len(f) for f in frames_pos_use]
print(f"3. Particle count stability")
print(f"   Min particles in any frame : {min(counts)}")
print(f"   Max particles in any frame : {max(counts)}")
print(f"   {'✅ Stable' if min(counts) == max(counts) else '⚠️  Particle count varies — lost atoms!'}\n")

# 4. RDF peak location
rdf_final = rdf_history[-1]
peak_idx  = np.argmax(rdf_final)
peak_r    = r_centers[peak_idx]
peak_val  = rdf_final[peak_idx]
print(f"4. RDF peak")
print(f"   Peak location : r = {peak_r:.3f} λ_D")
print(f"   Peak value    : g(r) = {peak_val:.3f}")
print(f"   {'✅ Clear screening signature' if peak_val > 1.2 else '⚠️  Weak or no screening detected'}")

=== Simulation Sanity Checks ===

1. Avg electron-proton nearest distance : 1.044 λ_D
   Expected (screened)                  : < 1.0 λ_D
   ✅ Looks good

2. Coupling parameter Γ                 : 0.1658
   Γ << 1 → weakly coupled (Debye-Hückel valid)
   Γ >> 1 → strongly coupled (different regime)
   ✅ Weakly coupled — Debye screening applies

3. Particle count stability
   Min particles in any frame : 300
   Max particles in any frame : 300
   ✅ Stable

4. RDF peak
   Peak location : r = 0.122 λ_D
   Peak value    : g(r) = 2.504
   ✅ Clear screening signature
